# Colab 17a — Fair-eval re-run: alphabet-matched 2×2 retrieval matrix

Self-contained re-eval of the colab16b (AA-train) and colab17 (SS-train) encoders on
**alphabet-matched** eval sets. Fixes the eval-set framing error caught 2026-05-21:
prior SS-retrieval numbers used the 5 high-**AA** pair list, which measures
"find AA-partner in SS space" — not SS-Lev approximation. Correct rule:

> The eval set's high-sim partners are defined by `normLev ≥ 0.70` **in the alphabet
> of the input being encoded**, never by the encoder's training alphabet or by
> any biological criterion.

This notebook trains both encoders from scratch (same setups as the prior notebooks),
builds two eval sets, runs the 2×2 retrieval matrix below, and prints the headline
table next to the old (wrong-eval) numbers for direct comparison.

|                     | Feed AA → high-AA eval | Feed SS → high-SS eval |
|---------------------|------------------------|------------------------|
| Encoder A (AA-train)| in-domain (sanity)     | **cross-rep AA→SS (new)** |
| Encoder B (SS-train)| **cross-rep SS→AA (sanity ≈ colab17 8/10)** | **in-domain SS (key new cell)** |

Key principle: the eval set is determined by the input alphabet, never by what the
encoder was trained on. Encoder choice ⊥ eval set choice.

No training-design changes from colab16b / colab17 — pure eval-side fix.


## 1. Setup

In [ ]:
import os
os.chdir('/content')
!rm -rf thesis-edit-distance-nn
!git clone https://github.com/katzemelli/thesis-edit-distance-nn.git
os.chdir('/content/thesis-edit-distance-nn')


In [ ]:
DATA_DIR = '/content/thesis-edit-distance-nn/sampledata/cath'
import os
for f in ['cath_s20_train70.csv.gz', 'cath_s20_test30.csv.gz', 'cath_eval.csv.gz']:
    p = os.path.join(DATA_DIR, f)
    print(f'{"OK" if os.path.exists(p) else "MISSING":<8} {p}')


In [ ]:
!pip install torch torchvision rapidfuzz scikit-learn --quiet


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from rapidfuzz.distance import Levenshtein as RFLevenshtein

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## 2. Constants

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
SS_ALPHABET = 'HLS'
VOCAB_SIZE = len(AA_ALPHABET) + 1
PAD_IDX = len(AA_ALPHABET)
AA_SET = set(AA_ALPHABET)
SS_SET = set(SS_ALPHABET)
CHAR_TO_IDX = {c: i for i, c in enumerate(AA_ALPHABET)}

MIN_LEN = 50
MAX_LEN_SEQ = 200
MAX_LEN = 200

N_TRAIN = 30000
NUM_EPOCHS = 30
BATCH_SIZE = 128
K = 16

# Two BAND_LOW values — one per encoder.
# AA mirrors colab16b: 20-letter alphabet, perturbation floor ~0.28 → BAND_LOW=0.30.
# SS mirrors colab17 actual run: 3-letter alphabet, perturbation floor ~0.51 → BAND_LOW=0.56.
BAND_LOW_AA = 0.30
BAND_LOW_SS = 0.56
BAND_HIGH   = 0.70

BIN_NAMES = ['far', 'mid', 'high']
BIN_MIDPOINTS = np.array([0.20, 0.55, 0.85])

print(f'AA encoder: BAND_LOW={BAND_LOW_AA}, alphabet=AA ({len(AA_ALPHABET)} letters)')
print(f'SS encoder: BAND_LOW={BAND_LOW_SS}, alphabet=SS ({len(SS_ALPHABET)} letters)')
print(f'BAND_HIGH (shared): {BAND_HIGH}')
print(f'K (shared headline): {K}')


## 3. Helpers — Levenshtein, encode, perturb, bin labelling

In [ ]:
def norm_lev(a, b):
    L = max(len(a), len(b))
    return 1.0 if L == 0 else 1.0 - RFLevenshtein.distance(a, b) / L

def is_standard_aa(seq): return all(c in AA_SET for c in seq)
def is_standard_ss(seq): return all(c in SS_SET for c in seq)

def encode_pad(seq, max_len=MAX_LEN, pad_idx=PAD_IDX):
    idx = [CHAR_TO_IDX[c] for c in seq][:max_len]
    idx += [pad_idx] * (max_len - len(idx))
    return torch.tensor(idx, dtype=torch.long)

def perturb(seq, k, alphabet, max_len=MAX_LEN):
    s = list(seq); abc = list(alphabet)
    for _ in range(k):
        if len(s) == 0: op = 'ins'
        elif len(s) >= max_len: op = rng.choice(['sub', 'del'])
        else: op = rng.choice(['sub', 'ins', 'del'])
        if op == 'sub':
            i = rng.integers(0, len(s))
            choices = [c for c in abc if c != s[i]]
            s[i] = rng.choice(choices)
        elif op == 'ins':
            i = rng.integers(0, len(s) + 1)
            s.insert(i, rng.choice(abc))
        elif op == 'del':
            i = rng.integers(0, len(s))
            del s[i]
    return ''.join(s)

def random_aa_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(AA_ALPHABET), size=length))

def random_ss_seq(min_len=MIN_LEN, max_len=MAX_LEN_SEQ):
    length = int(rng.integers(min_len, max_len + 1))
    return ''.join(rng.choice(list(SS_ALPHABET), size=length))

def bin_idx_for(x, band_low):
    if x < band_low: return 0
    if x < BAND_HIGH: return 1
    return 2

def bin_names_arr_for(xs, band_low):
    xs = np.asarray(xs)
    out = np.ones(xs.shape, dtype=np.int64)   # mid default
    out[xs < band_low]   = 0
    out[xs >= BAND_HIGH] = 2
    return np.array(BIN_NAMES)[out]


def make_targeted_perturbation_pairs(seed_fn, alphabet, n_pairs):
    pairs = []
    attempts = 0
    max_attempts = n_pairs * 4
    while len(pairs) < n_pairs and attempts < max_attempts:
        attempts += 1
        seed = seed_fn()
        if seed is None or len(seed) < MIN_LEN: continue
        L = len(seed)
        t = float(rng.uniform(0.0, 1.0))
        k = max(0, int(round((1.0 - t) * L)))
        other = perturb(seed, k, alphabet)
        if 1 <= len(other) <= MAX_LEN:
            label = norm_lev(seed, other)
            pairs.append((seed, other, label))
    return pairs


## 4. Architecture — encoder (AdaptiveAvgPool) + classification head

In [ ]:
class SiameseEncoder(nn.Module):
    def __init__(self, K, vocab_size=VOCAB_SIZE, embed_dim=32,
                 hidden1=32, hidden2=64, out_dim=128, pad_idx=PAD_IDX):
        super().__init__()
        self.K = K
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.conv1 = nn.Conv1d(embed_dim, hidden1, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(hidden1, hidden2, kernel_size=3, padding=1)
        self.pool  = nn.AdaptiveAvgPool1d(K)
        self.fc    = nn.Linear(hidden2 * K, out_dim)

    def forward(self, x):
        mask = (x != self.pad_idx).float()
        e = self.embedding(x).permute(0, 2, 1)
        h = F.relu(self.conv1(e))
        h = F.relu(self.conv2(h))
        h = h * mask.unsqueeze(1)
        h = self.pool(h)
        h = h.flatten(1)
        z = self.fc(h)
        return F.normalize(z, p=2, dim=1)


class SiameseClassifier(nn.Module):
    def __init__(self, K, embed_out=128, hidden_mlp=64, n_bins=3):
        super().__init__()
        self.encoder = SiameseEncoder(K)
        self.head = nn.Sequential(
            nn.Linear(embed_out, hidden_mlp),
            nn.LeakyReLU(0.01),
            nn.Linear(hidden_mlp, n_bins),
        )

    def forward(self, a, b):
        e_a = self.encoder(a)
        e_b = self.encoder(b)
        return self.head(torch.abs(e_a - e_b))

    def encode(self, x):
        return self.encoder(x)


## 5. Pair dataset — parameterised by band_low

In [ ]:
class PairDatasetCE(Dataset):
    def __init__(self, pairs, band_low):
        self.a_encoded = [encode_pad(a) for a, _, _ in pairs]
        self.b_encoded = [encode_pad(b) for _, b, _ in pairs]
        self.bins = torch.tensor(
            [bin_idx_for(label, band_low) for _, _, label in pairs],
            dtype=torch.long,
        )

    def __len__(self): return len(self.bins)
    def __getitem__(self, i):
        return self.a_encoded[i], self.b_encoded[i], self.bins[i]


## 6. Training function (alphabet + band_low parameterised)

In [ ]:
def train_encoder(seed_fn, alphabet, band_low, label, n_epochs=NUM_EPOCHS, seed=SEED):
    # Reset seeds so AA and SS runs are independent and reproducible
    torch.manual_seed(seed)
    np.random.seed(seed)

    print(f'\n========== Training {label} encoder ==========')
    print(f'  alphabet={alphabet}  band_low={band_low}  K={K}  n_pairs={N_TRAIN}  epochs={n_epochs}')

    pairs = make_targeted_perturbation_pairs(seed_fn=seed_fn, alphabet=alphabet, n_pairs=N_TRAIN)
    labels = np.array([p[2] for p in pairs])
    bin_arr = bin_names_arr_for(labels, band_low)
    bin_counts = {b: int((bin_arr == b).sum()) for b in BIN_NAMES}
    print(f'  generated {len(pairs)} pairs, label range [{labels.min():.3f}, {labels.max():.3f}]')
    print(f'  bin counts at band_low={band_low}: {bin_counts}')

    ds = PairDatasetCE(pairs, band_low=band_low)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True)

    model = SiameseClassifier(K).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    losses = []
    model.train()
    for epoch in range(1, n_epochs + 1):
        ep_loss = 0.0; nb = 0
        for a, b, bins in dl:
            a, b, bins = a.to(device), b.to(device), bins.to(device)
            loss = criterion(model(a, b), bins)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            ep_loss += loss.item(); nb += 1
        losses.append(ep_loss / nb)
        if epoch % 5 == 0 or epoch == 1:
            print(f'  [{label}] epoch {epoch:3d}/{n_epochs} — CE: {losses[-1]:.4f}')
    return model, losses


## 7. Data load — eval set + protein pool

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_train70.csv.gz'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'cath_s20_test30.csv.gz'))
prot_df = pd.concat([train_df, test_df], ignore_index=True).drop_duplicates('domain_id')

prot_df = prot_df[prot_df['aa_seq'].apply(is_standard_aa)]
prot_df = prot_df[prot_df['ss_seq'].apply(is_standard_ss)]
prot_df = prot_df[prot_df['aa_seq'].str.len() == prot_df['ss_seq'].str.len()]
prot_df['aa_len'] = prot_df['aa_seq'].str.len()

RESCUED = {'4z0mC02', '3qkaE02'}
in_range = prot_df['aa_len'].between(MIN_LEN, MAX_LEN)
prot_df = prot_df[in_range | prot_df['domain_id'].isin(RESCUED)].reset_index(drop=True)

id_to_aa = dict(zip(prot_df['domain_id'], prot_df['aa_seq']))
id_to_ss = dict(zip(prot_df['domain_id'], prot_df['ss_seq']))
all_domains = list(prot_df['domain_id'])
pool_set = set(all_domains)
print(f'Protein pool: {len(prot_df)} (incl. {len(prot_df) - in_range.sum()} rescued short)')

eval_df = pd.read_csv(os.path.join(DATA_DIR, 'cath_eval.csv.gz'))
print(f'Eval pairs: {len(eval_df)}')


## 8. Build TWO alphabet-matched eval sets

- **Set α (high-AA):** all pairs with `aa_bin == 'high'` (i.e. `aa_score ≥ 0.70`). 5 pairs.
  Same set we've been using since colab15.
- **Set σ (high-SS):** top-5 by `ss_score` among pairs where both proteins are in the
  pool and neither protein appears in Set α (independence sanity).

The two sets test Lev-approximation on different alphabets. They share no proteins, so
performance on one set tells us nothing about performance on the other by construction.


In [ ]:
# Set α — high-AA pairs (existing)
HIGH_AA_PAIRS = eval_df[eval_df['aa_bin'] == 'high'][['domain_a', 'domain_b']].values.tolist()
high_aa_proteins = {d for pair in HIGH_AA_PAIRS for d in pair}
print(f'Set α (high-AA, n={len(HIGH_AA_PAIRS)}):')
for a, b in HIGH_AA_PAIRS:
    row = eval_df[(eval_df['domain_a']==a) & (eval_df['domain_b']==b)].iloc[0]
    print(f'  {a} ↔ {b}   aa_score={row["aa_score"]:.3f}  ss_score={row["ss_score"]:.3f}')

# Set σ — top-5 by ss_score, both in pool, no protein overlap with Set α
cand = eval_df[eval_df['ss_score'] >= BAND_HIGH].copy()
cand = cand[cand['domain_a'].isin(pool_set) & cand['domain_b'].isin(pool_set)]
cand = cand[~cand['domain_a'].isin(high_aa_proteins) & ~cand['domain_b'].isin(high_aa_proteins)]
cand = cand.sort_values('ss_score', ascending=False)
HIGH_SS_PAIRS = cand.head(5)[['domain_a', 'domain_b']].values.tolist()
high_ss_proteins = {d for pair in HIGH_SS_PAIRS for d in pair}

print(f'\nSet σ (high-SS, n={len(HIGH_SS_PAIRS)}):')
for a, b in HIGH_SS_PAIRS:
    row = eval_df[(eval_df['domain_a']==a) & (eval_df['domain_b']==b)].iloc[0]
    print(f'  {a} ↔ {b}   aa_score={row["aa_score"]:.3f}  ss_score={row["ss_score"]:.3f}')

# Sanity: disjoint protein sets?
overlap = high_aa_proteins & high_ss_proteins
print(f'\nProtein overlap between Set α and Set σ: {len(overlap)} (should be 0)')

# Cross-score diagnostic: how similar are these pairs in the OTHER alphabet?
print('\nCross-score check (confirms sets really do test different functions):')
hi_aa_pairs_ss = [eval_df[(eval_df["domain_a"]==a)&(eval_df["domain_b"]==b)]["ss_score"].iloc[0]
                  for a, b in HIGH_AA_PAIRS]
hi_ss_pairs_aa = [eval_df[(eval_df["domain_a"]==a)&(eval_df["domain_b"]==b)]["aa_score"].iloc[0]
                  for a, b in HIGH_SS_PAIRS]
print(f'  Set α (high-AA) — ss_score range: [{min(hi_aa_pairs_ss):.3f}, {max(hi_aa_pairs_ss):.3f}], mean={np.mean(hi_aa_pairs_ss):.3f}')
print(f'  Set σ (high-SS) — aa_score range: [{min(hi_ss_pairs_aa):.3f}, {max(hi_ss_pairs_aa):.3f}], mean={np.mean(hi_ss_pairs_aa):.3f}')


## 9. Train AA encoder (mirror colab16b)

In [ ]:
model_aa, losses_aa = train_encoder(
    seed_fn=random_aa_seq, alphabet=AA_ALPHABET,
    band_low=BAND_LOW_AA, label='AA',
)


## 10. Train SS encoder (mirror colab17)

In [ ]:
model_ss, losses_ss = train_encoder(
    seed_fn=random_ss_seq, alphabet=SS_ALPHABET,
    band_low=BAND_LOW_SS, label='SS',
)


## 11. Retrieval helper — alphabet-agnostic

In [ ]:
@torch.no_grad()
def encode_pool(model, seq_lookup, ids, batch_size=256):
    model.eval()
    out = []
    for i in range(0, len(ids), batch_size):
        chunk = ids[i:i+batch_size]
        x = torch.stack([encode_pad(seq_lookup[d]) for d in chunk]).to(device)
        out.append(model.encoder(x))
    return torch.cat(out, 0)


@torch.no_grad()
def head_probs(model, e_query, pool_emb, batch_size=4096):
    N = pool_emb.shape[0]
    probs = torch.empty((N, 3), device=device)
    e_q = e_query.unsqueeze(0)
    for i in range(0, N, batch_size):
        diff = torch.abs(pool_emb[i:i+batch_size] - e_q)
        probs[i:i+batch_size] = F.softmax(model.head(diff), dim=1)
    return probs


def run_retrieval(model, seq_lookup, eval_pairs, label):
    """Run k-NN retrieval for a list of (a_id, b_id) eval pairs.

    `seq_lookup` chooses the input alphabet (id_to_aa or id_to_ss).
    Reports hits@{1,5,10,50} under L2-sim, P(high), and E[bin_midpoint] scores.
    """
    model.eval()
    pool_emb = encode_pool(model, seq_lookup, all_domains)
    id_to_idx = {d: i for i, d in enumerate(all_domains)}
    K_RECALL = (1, 5, 10, 50)
    METRIC_NAMES = ['L2', 'P(high)', 'E[bin_mid]']
    results = {m: {'ranks': [], 'hits': {k: 0 for k in K_RECALL}} for m in METRIC_NAMES}

    print(f'\n--- {label} (pool={len(all_domains)}) ---')
    print(f'{"query":<10} {"partner":<10}  {"rank_L2":>8} {"rank_Phigh":>11} {"rank_Emid":>10}')
    for a, b in eval_pairs:
        for q, partner in [(a, b), (b, a)]:
            if q not in id_to_idx or partner not in id_to_idx: continue
            qi, pi = id_to_idx[q], id_to_idx[partner]
            e_q = pool_emb[qi]
            l2_sim = (1.0 - torch.linalg.vector_norm(pool_emb - e_q, ord=2, dim=1) / 2.0
                      ).cpu().numpy()
            l2_sim[qi] = -np.inf
            rank_l2 = int(np.where(np.argsort(-l2_sim) == pi)[0][0]) + 1

            probs = head_probs(model, e_q, pool_emb).cpu().numpy()
            p_hi = probs[:, 2]
            emid = (probs * BIN_MIDPOINTS[None, :]).sum(axis=1)
            p_hi[qi] = -np.inf; emid[qi] = -np.inf
            rank_phigh = int(np.where(np.argsort(-p_hi) == pi)[0][0]) + 1
            rank_emid  = int(np.where(np.argsort(-emid) == pi)[0][0]) + 1

            for m_name, r in [('L2', rank_l2), ('P(high)', rank_phigh), ('E[bin_mid]', rank_emid)]:
                results[m_name]['ranks'].append((q, partner, r))
                for k in K_RECALL:
                    if r <= k: results[m_name]['hits'][k] += 1
            print(f'{q:<10} {partner:<10}  {rank_l2:>8} {rank_phigh:>11} {rank_emid:>10}')

    print(f'\n  Hits@k summary:')
    print(f'  {"metric":<12} {"@1":>6} {"@5":>6} {"@10":>6} {"@50":>6}')
    for m in METRIC_NAMES:
        nq = len(results[m]['ranks'])
        h = results[m]['hits']
        print(f'  {m:<12} {h[1]:>3}/{nq:<2} {h[5]:>3}/{nq:<2} {h[10]:>3}/{nq:<2} {h[50]:>3}/{nq:<2}')
    return results


## 12. 2×2 retrieval matrix

Four cells. Encoder choice ⊥ eval set choice. Rule: the alphabet fed in determines
which eval set is used (`id_to_aa` ↔ `HIGH_AA_PAIRS`, `id_to_ss` ↔ `HIGH_SS_PAIRS`).


In [ ]:
matrix = {}
# Encoders × (input-alphabet, eval-set) combinations
configs = [
    ('AA-trained', model_aa, 'AA-feed (high-AA eval)', id_to_aa, HIGH_AA_PAIRS, 'in-domain AA'),
    ('AA-trained', model_aa, 'SS-feed (high-SS eval)', id_to_ss, HIGH_SS_PAIRS, 'cross-rep AA→SS'),
    ('SS-trained', model_ss, 'AA-feed (high-AA eval)', id_to_aa, HIGH_AA_PAIRS, 'cross-rep SS→AA'),
    ('SS-trained', model_ss, 'SS-feed (high-SS eval)', id_to_ss, HIGH_SS_PAIRS, 'in-domain SS'),
]

for enc_label, model, feed_label, lookup, pairs, role in configs:
    label = f'{enc_label} × {feed_label}  [{role}]'
    matrix[(enc_label, feed_label)] = {
        'role': role,
        'result': run_retrieval(model, lookup, pairs, label),
    }


## 13. Headline 2×2 table — hits@10 (L2) per cell, vs OLD wrong-eval numbers

In [ ]:
print('=' * 78)
print('FAIR-EVAL 2x2 — hits@10 (L2) per cell')
print('=' * 78)
print()
print(f'{"":<14} | {"Feed AA → high-AA eval":^26} | {"Feed SS → high-SS eval":^26}')
print('-' * 78)
for enc in ['AA-trained', 'SS-trained']:
    cells_str = []
    for feed in ['AA-feed (high-AA eval)', 'SS-feed (high-SS eval)']:
        cell = matrix[(enc, feed)]
        h10 = cell['result']['L2']['hits'][10]
        nq = len(cell['result']['L2']['ranks'])
        cells_str.append(f'{cell["role"]:<14}  {h10}/{nq:<2}')
    print(f'{enc:<14} | {cells_str[0]:^26} | {cells_str[1]:^26}')

print()
print('OLD (wrong-eval) numbers for comparison — both SS-side numbers below used')
print('the high-AA pair list, which measures "find AA-partner in SS space".')
print()
print('  colab16b (AA-train, K=16):')
print('    AA in-domain (high-AA eval, RIGHT):       hits@10 (L2) = 10/10')
print('    SS cross-rep (high-AA eval, WRONG):       hits@10 (L2) =  3/10   (P_high=5/10)')
print('  colab17  (SS-train, K=16):')
print('    AA cross-rep (high-AA eval, RIGHT):       hits@10 (L2) =  8/10')
print('    SS in-domain (high-AA eval, WRONG):       hits@10 (L2) =  3/10   (P_high=5/10)')
print()
print('The two RIGHT numbers (in-domain AA, cross-rep SS→AA on high-AA eval) should')
print('roughly reproduce above. The two cells with high-SS eval are NEW.')


## 14. Per-pair breakdown for each of the four cells

In [ ]:
for enc, feed in [
    ('AA-trained', 'AA-feed (high-AA eval)'),
    ('AA-trained', 'SS-feed (high-SS eval)'),
    ('SS-trained', 'AA-feed (high-AA eval)'),
    ('SS-trained', 'SS-feed (high-SS eval)'),
]:
    cell = matrix[(enc, feed)]
    print(f'\n=== {enc} × {feed}  [{cell["role"]}] ===')
    rows = cell['result']['L2']['ranks']
    rows_ph = cell['result']['P(high)']['ranks']
    rows_em = cell['result']['E[bin_mid]']['ranks']
    print(f'  {"query":<10} {"partner":<10}  {"L2":>6} {"Phigh":>6} {"Emid":>6}')
    for (q, p, r_l2), (_, _, r_ph), (_, _, r_em) in zip(rows, rows_ph, rows_em):
        print(f'  {q:<10} {p:<10}  {r_l2:>6} {r_ph:>6} {r_em:>6}')


## 15. Interpretation — three reads on the fair-eval matrix

This block prints the three candidate readings of the matrix and which numbers
support each. Final interpretation is for Melissa to confirm in conversation.


In [ ]:
h_aa_indomain  = matrix[('AA-trained', 'AA-feed (high-AA eval)')]['result']['L2']['hits'][10]
h_aa_cross_ss  = matrix[('AA-trained', 'SS-feed (high-SS eval)')]['result']['L2']['hits'][10]
h_ss_cross_aa  = matrix[('SS-trained', 'AA-feed (high-AA eval)')]['result']['L2']['hits'][10]
h_ss_indomain  = matrix[('SS-trained', 'SS-feed (high-SS eval)')]['result']['L2']['hits'][10]

print('=' * 78)
print('INTERPRETATION — three reads on the fair-eval 2x2')
print('=' * 78)
print()
print(f'  AA in-domain         (sanity):  {h_aa_indomain}/10')
print(f'  cross-rep AA→SS (fair, NEW):    {h_aa_cross_ss}/10')
print(f'  cross-rep SS→AA (sanity):       {h_ss_cross_aa}/10')
print(f'  SS in-domain    (fair, NEW):    {h_ss_indomain}/10   <-- key cell')
print()

# Heuristic reads
THRESH_STRONG = 7
THRESH_WEAK   = 4

if h_ss_indomain >= THRESH_STRONG and h_aa_cross_ss >= THRESH_STRONG:
    print('READ: SYMMETRIC ZERO-SHOT TRANSFER')
    print('  Both in-domain cells strong AND both cross-rep cells strong.')
    print('  Encoder learns Lev approximation on each alphabet AND transfers zero-shot.')
    print('  Headline for writeup: symmetric Lev-approximation across alphabets.')
elif h_ss_indomain >= THRESH_STRONG and h_aa_cross_ss < THRESH_WEAK:
    print('READ: SS LEARNS BUT DOES NOT TRANSFER FROM AA')
    print('  SS in-domain works (encoder can approximate SS-Lev when trained on SS),')
    print('  but AA-trained encoder fails to approximate SS-Lev zero-shot.')
    print('  AA→SS transfer is alphabet-specific rather than function-general.')
elif h_ss_indomain < THRESH_WEAK and h_aa_cross_ss < THRESH_WEAK:
    print('READ: SS-LEV APPROXIMATION INTRINSICALLY HARD')
    print('  SS in-domain weak AND cross-rep AA→SS weak — the 3-letter alphabet')
    print('  is the bottleneck, not the eval-set choice. Worth pivoting to 8-letter')
    print('  DSSP or revisiting the SS training pipeline (BAND_LOW, perturbation depth).')
else:
    print('READ: MIXED — see per-pair breakdown above for which queries fail')
    print('  No clean single read. Likely a few specific pairs (length-mismatched,')
    print('  short sequences) drag the average. Examine per-pair ranks before next step.')


## 16. Summary block for BENCHMARKS.md

In [ ]:
print('=' * 80)
print('COLAB17A SUMMARY (fair-eval re-run, alphabet-matched 2x2) — for BENCHMARKS.md')
print('=' * 80)
print(f'K = {K}, AA encoder BAND_LOW = {BAND_LOW_AA}, SS encoder BAND_LOW = {BAND_LOW_SS}')
print(f'Eval Set α (high-AA): n=5 pairs, aa_score >= 0.70')
print(f'Eval Set σ (high-SS): n=5 pairs, top-5 by ss_score (clean of Set α proteins)')
print()
print(f'  {"":>14} | {"high-AA eval (L2)":>20} | {"high-SS eval (L2)":>20}')
print(f'  {"-"*14}-+-{"-"*20}-+-{"-"*20}')
for enc in ['AA-trained', 'SS-trained']:
    h_aa = matrix[(enc, 'AA-feed (high-AA eval)')]['result']['L2']['hits'][10]
    nq_aa = len(matrix[(enc, 'AA-feed (high-AA eval)')]['result']['L2']['ranks'])
    h_ss = matrix[(enc, 'SS-feed (high-SS eval)')]['result']['L2']['hits'][10]
    nq_ss = len(matrix[(enc, 'SS-feed (high-SS eval)')]['result']['L2']['ranks'])
    print(f'  {enc:>14} | {f"hits@10 = {h_aa}/{nq_aa}":>20} | {f"hits@10 = {h_ss}/{nq_ss}":>20}')
